<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 8: K-Means Müşteri Segmentasyonu

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 8 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta08/hafta08_kmeans_segmentasyon.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta08/hafta08_kmeans_segmentasyon.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>&nbsp;
<a href="https://raw.githubusercontent.com/DrMuratAltun/VB-YZ-90/main/web/public/sunumlar/hafta08_kumeleme_oneri.pdf"><img src="https://img.shields.io/badge/PDF%20Sunum-EC1C24?style=flat&logo=adobeacrobatreader&logoColor=white" alt="PDF Sunum"/></a>&nbsp;
<a href="https://drmurataltun.github.io/VB-YZ-90/hafta/08/"><img src="https://img.shields.io/badge/Web%20Sitesi-2B7A78?style=flat&logo=googlechrome&logoColor=white" alt="Web Sitesi"/></a>

</div>

---

**Eğitmen:** Dr. Murat Altun · [yapayzekaokulum.com](https://yapayzekaokulum.com) · [GitHub](https://github.com/DrMuratAltun)

**Program:** ECS Veri Bilimi ve Yapay Zeka Uzmanlığı · 90 Saat · 15 Hafta
---

> **Bu defterde neler öğreneceksiniz?**
>
> - K-Means kümeleme algoritması
> - Elbow Method ve Silhouette Score
> - RFM müşteri segmentasyonu

# Hafta 8 - Müşteri Segmentasyonu (KMeans Kümeleme)

Bu derste:
- Sentetik müşteri verisi oluşturacağız
- Keşifsel veri analizi (EDA) yapacağız
- KMeans kümeleme algoritmasını uygulayacağız
- Dirsek (Elbow) yöntemi ile optimal küme sayısını belirleyeceğiz
- Silhouette skoru ile değerlendirme yapacağız
- Müşteri segmentlerini isimlendirip iş önerileri çıkaracağız

## 1. Kütüphanelerin Yüklenmesi

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 2. Gerçek Mall Customers Verisini Yükleme

**Mall Customers Dataset** — Bir alışveriş merkezinin 200 müşterisinin demografik bilgileri ve harcama skorları. KMeans kümeleme için klasik referans veri setidir.

**Kaynak:** [Kaggle - Mall Customer Segmentation](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)

**Sütunlar:** CustomerID, Gender, Age, Annual Income (k$), Spending Score (1-100)

In [ ]:
# Mall Customers Dataset (gerçek veri)
try:
    url = "https://raw.githubusercontent.com/tirthajyoti/Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv"
    df = pd.read_csv(url)
except:
    url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/Mall_Customers.csv"
    df = pd.read_csv(url)

# Sütun isimlerini kısaltalım
df.columns = ['CustomerID', 'Gender', 'Age', 'AnnualIncome', 'SpendingScore']

print(f"Veri seti boyutu: {df.shape}")
print(f"\nSütunlar: {list(df.columns)}")
print(f"\nCinsiyet dağılımı:")
print(df['Gender'].value_counts())
print(f"\nTemel istatistikler:")
df.describe()

## 3. Keşifsel Veri Analizi (EDA)

### Temel İstatistikler

Verinin genel yapısını inceliyoruz: sütun tipleri, eksik değerler, temel istatistikler (ortalama, medyan, min, max). Bu bilgiler veri temizleme ve ön işleme adımlarını planlamak için gereklidir.

In [ ]:
df.describe()

### Temel İstatistikler

Verinin genel yapısını inceliyoruz: sütun tipleri, eksik değerler, temel istatistikler (ortalama, medyan, min, max). Bu bilgiler veri temizleme ve ön işleme adımlarını planlamak için gereklidir.

In [ ]:
df.info()

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
# Cinsiyet dağılımı
print("Cinsiyet dağılımı:")
print(df['Gender'].value_counts())

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['Age'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Yaş Dağılımı')
axes[0].set_xlabel('Yaş')
axes[0].set_ylabel('Frekans')

axes[1].hist(df['AnnualIncome'], bins=20, color='seagreen', edgecolor='white')
axes[1].set_title('Yıllık Gelir Dağılımı (bin TL)')
axes[1].set_xlabel('Yıllık Gelir')
axes[1].set_ylabel('Frekans')

axes[2].hist(df['SpendingScore'], bins=20, color='coral', edgecolor='white')
axes[2].set_title('Harcama Skoru Dağılımı')
axes[2].set_xlabel('Harcama Skoru')
axes[2].set_ylabel('Frekans')

plt.tight_layout()
plt.show()

### Saçılım Grafiği

İki değişken arasındaki ilişkiyi saçılım grafiği ile inceliyoruz. Noktaların oluşturduğu desen, doğrusal veya doğrusal olmayan ilişkiyi gösterir.

In [ ]:
# Scatter plot: Gelir vs Harcama Skoru
plt.figure(figsize=(10, 7))
plt.scatter(df['AnnualIncome'], df['SpendingScore'], 
            c='steelblue', alpha=0.6, edgecolors='white', s=60)
plt.xlabel('Yıllık Gelir (bin TL)')
plt.ylabel('Harcama Skoru')
plt.title('Yıllık Gelir vs Harcama Skoru')
plt.show()

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Korelasyon matrisi
numeric_cols = df[['Age', 'AnnualIncome', 'SpendingScore']]
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Korelasyon Matrisi')
plt.show()

## 4. Veri Ön İşleme

KMeans algoritması uzaklık tabanlı olduğu için, özellikleri ölçeklendirmemiz gerekir.

In [ ]:
# Kümeleme için kullanılacak özellikler
X = df[['AnnualIncome', 'SpendingScore']].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Ölçeklendirilmiş verinin ilk 5 satırı:")
print(X_scaled[:5])

## 5. Dirsek (Elbow) Yöntemi

Optimal küme sayısını belirlemek için farklı k değerlerinde **inertia** (küme içi kareler toplamı) değerlerini hesaplıyoruz.

In [ ]:
inertias = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Küme Sayısı (k)')
plt.ylabel('Inertia (Küme İçi Kareler Toplamı)')
plt.title('Dirsek (Elbow) Yöntemi')
plt.xticks(K_range)
plt.grid(True)
plt.show()

## 6. Silhouette Skoru

Silhouette skoru, kümeleme kalitesini -1 ile 1 arasında ölçer:
- **1'e yakın**: İyi kümelenmiş
- **0'a yakın**: Kümeler arası örtüşme var
- **Negatif**: Yanlış kümeye atanmış

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"k={k}: Silhouette Skoru = {score:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(range(2, 11), silhouette_scores, 'go-', linewidth=2, markersize=8)
plt.xlabel('Küme Sayısı (k)')
plt.ylabel('Silhouette Skoru')
plt.title('Silhouette Skoru vs Küme Sayısı')
plt.xticks(range(2, 11))
plt.grid(True)
plt.show()

optimal_k = range(2, 11)[np.argmax(silhouette_scores)]
print(f"\nOptimal küme sayısı: {optimal_k}")

## 7. Final Model: k=5 ile KMeans

Müşteri segmentasyonu için 5 küme seçiyoruz (iş mantığına uygun).

In [ ]:
k = 5
kmeans_final = KMeans(n_clusters=k, random_state=42, n_init=10)
df['Cluster'] = kmeans_final.fit_predict(X_scaled)

print("Küme dağılımı:")
print(df['Cluster'].value_counts().sort_index())

### Kümeleme Analizi

K-Means algoritması ile veriyi doğal gruplara ayırıyoruz. Optimal küme sayısını belirlemek için Elbow yöntemi ve Silhouette skoru kullanıyoruz.

In [ ]:
# Küme merkezlerini orijinal ölçeğe geri dönüştür
centers_original = scaler.inverse_transform(kmeans_final.cluster_centers_)
print("\nKüme Merkezleri (Orijinal Ölçek):")
centers_df = pd.DataFrame(centers_original, columns=['AnnualIncome', 'SpendingScore'])
centers_df.index.name = 'Küme'
centers_df

## 8. Kümelerin Görselleştirilmesi

### Saçılım Grafiği

İki değişken arasındaki ilişkiyi saçılım grafiği ile inceliyoruz. Noktaların oluşturduğu desen, doğrusal veya doğrusal olmayan ilişkiyi gösterir.

In [ ]:
colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

plt.figure(figsize=(12, 8))
for i in range(k):
    cluster_data = df[df['Cluster'] == i]
    plt.scatter(cluster_data['AnnualIncome'], cluster_data['SpendingScore'],
                c=colors[i], label=f'Küme {i}', alpha=0.7, edgecolors='white', s=80)

plt.scatter(centers_original[:, 0], centers_original[:, 1],
            c='black', marker='X', s=200, label='Küme Merkezleri', zorder=5)

plt.xlabel('Yıllık Gelir (bin TL)', fontsize=13)
plt.ylabel('Harcama Skoru', fontsize=13)
plt.title('Müşteri Segmentasyonu (KMeans k=5)', fontsize=15)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## 9. Segment İsimlendirme ve İş Önerileri

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
# Her kümenin istatistiklerini inceleyelim
cluster_summary = df.groupby('Cluster').agg({
    'Age': 'mean',
    'AnnualIncome': 'mean',
    'SpendingScore': 'mean',
    'CustomerID': 'count'
}).rename(columns={'CustomerID': 'Müşteri Sayısı'})

cluster_summary = cluster_summary.round(1)
cluster_summary

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
# Segment isimleri atama (küme özelliklerine göre)
segment_names = {
    0: 'VIP',
    1: 'Sadık',
    2: 'Riskli',
    3: 'Tutumlu',
    4: 'Potansiyel'
}

df['Segment'] = df['Cluster'].map(segment_names)
print("Segment dağılımı:")
print(df['Segment'].value_counts())

### Saçılım Grafiği

İki değişken arasındaki ilişkiyi saçılım grafiği ile inceliyoruz. Noktaların oluşturduğu desen, doğrusal veya doğrusal olmayan ilişkiyi gösterir.

In [ ]:
# Segmentlere göre görselleştirme
plt.figure(figsize=(12, 8))
for i, (cluster_id, name) in enumerate(segment_names.items()):
    cluster_data = df[df['Cluster'] == cluster_id]
    plt.scatter(cluster_data['AnnualIncome'], cluster_data['SpendingScore'],
                c=colors[i], label=name, alpha=0.7, edgecolors='white', s=80)

plt.xlabel('Yıllık Gelir (bin TL)', fontsize=13)
plt.ylabel('Harcama Skoru', fontsize=13)
plt.title('Müşteri Segmentleri', fontsize=15)
plt.legend(fontsize=12, title='Segment')
plt.grid(True, alpha=0.3)
plt.show()

## 10. İş Önerileri

| Segment | Özellik | İş Önerisi |
|---------|---------|------------|
| **VIP** | Yüksek gelir, yüksek harcama | Özel sadakat programı, kişiselleştirilmiş teklifler, premium hizmetler |
| **Sadık** | Orta gelir, orta-yüksek harcama | Sadakat puanları, düzenli kampanyalar, çapraz satış fırsatları |
| **Riskli** | Düşük gelir, düşük harcama | Kaybetme riski yüksek; geri kazanım kampanyaları, indirim kuponları |
| **Tutumlu** | Yüksek gelir, düşük harcama | Harcamayı artırmaya yönelik özel teklifler, ürün önerileri |
| **Potansiyel** | Genç, büyüme potansiyeli | Marka bilinirliği artırma, giriş seviyesi kampanyalar, sosyal medya hedefleme |

## Özet

Bu derste öğrendiklerimiz:
- **KMeans** kümeleme algoritması ile müşteri segmentasyonu yapılabilir
- **Dirsek yöntemi** ve **Silhouette skoru** optimal küme sayısını belirlemede yardımcı olur
- Kümeleme sonuçlarını iş bağlamında yorumlamak kritik öneme sahiptir
- Her segment için farklı pazarlama stratejileri geliştirilebilir

### Alıştırma
1. Yaş ve gelir özelliklerini birlikte kullanarak kümelemeyi tekrar deneyin
2. k=3 ve k=7 için sonuçları karşılaştırın
3. Cinsiyet bazlı segment analizi yapın

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

&copy; 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>